In [14]:
import pandas as pd
import numpy as np

In [15]:
df = pd.read_csv("indicators_tun.csv")

In [16]:
df.head()

,Country Name,Country ISO3,Year,Indicator Name,Indicator Code,Value
0,#country+name,#country+code,#date+year,#indicator+name,#indicator+code,#indicator+value+num
1,Tunisia,TUN,2022,Fertilizer consumption (% of fertilizer produc...,AG.CON.FERT.PT.ZS,110.740734927371
2,Tunisia,TUN,2021,Fertilizer consumption (% of fertilizer produc...,AG.CON.FERT.PT.ZS,110.319806252816
3,Tunisia,TUN,2020,Fertilizer consumption (% of fertilizer produc...,AG.CON.FERT.PT.ZS,100.577963621517
4,Tunisia,TUN,2019,Fertilizer consumption (% of fertilizer produc...,AG.CON.FERT.PT.ZS,34.3912956024417


In [17]:
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84215 entries, 0 to 84214
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Country Name    84215 non-null  object
 1   Country ISO3    84215 non-null  object
 2   Year            84215 non-null  object
 3   Indicator Name  84215 non-null  object
 4   Indicator Code  84215 non-null  object
 5   Value           84215 non-null  object
dtypes: object(6)
memory usage: 3.9+ MB


In [18]:
df.describe()

,Country Name,Country ISO3,Year,Indicator Name,Indicator Code,Value
count,84215,84215,84215,84215,84215,84215
unique,2,2,66,2866,2867,53940
top,Tunisia,TUN,2010,"Total reserves (includes gold, current US$)",FI.RES.TOTL.CD,0
freq,84214,84214,2588,195,195,1917


In [19]:
df.columns 

Index(['Country Name', 'Country ISO3', 'Year', 'Indicator Name',
       'Indicator Code', 'Value'],
      dtype='object')

In [20]:
df.shape 

(84215, 6)

In [21]:
# View how many total duplicate values exist in the 'Indicator Name' column
duplicate_count = df["Indicator Name"].duplicated().sum()
print(f"🔁 Number of duplicate values in 'Indicator Name': {duplicate_count}")

# View number of unique values
unique_values_count = df["Indicator Name"].nunique()
print(f"🔢 Number of unique values in 'Indicator Name': {unique_values_count}")

# View frequency of each unique value (i.e., how often each occurs)
value_counts = df["Indicator Name"].value_counts()
print("\n📊 Frequency of each unique value in 'Indicator Name':")
print(value_counts)

# Optional: display top 10 most frequent indicators
print("\n🔥 Top 10 most frequent Indicator Names:")
print(value_counts.head(10))


🔁 Number of duplicate values in 'Indicator Name': 81349
🔢 Number of unique values in 'Indicator Name': 2866

📊 Frequency of each unique value in 'Indicator Name':
Indicator Name
Total reserves (includes gold, current US$)                                    195
School enrollment, primary and secondary (gross), gender parity index (GPI)    195
Net migration                                                                  195
Life expectancy at birth, male (years)                                         192
Life expectancy at birth, female (years)                                       192
                                                                              ... 
Coverage in 2nd quintile (%) - Unconditional Cash Transfers -rural               1
Coverage in 2nd quintile (%) - Unconditional Cash Transfers                      1
Coverage in 2nd quintile (%) - Unconditional Cash Transfers -urban               1
Coverage in 3rd quintile (%) - Unconditional Cash Transfers (preT)         

In [25]:
# Ensure only Tunisia's data is included
df = df[df['Country ISO3'] == 'TUN']

# Convert 'Year' column to integer to avoid type errors
df['Year'] = df['Year'].astype(int)

# Convert 'Value' column to numeric, handling malformed data
df['Value'] = pd.to_numeric(df['Value'], errors='coerce')

# Define the economic indicators mapping with French names
economic_indicators = {
    "Produit Intérieur Brut (aux prix du marché)": {
        "type": "direct",
        "code": "NY.GDP.MKTP.CD",
        "description": "Gross Domestic Product at market prices (current US$). Use NY.GDP.MKTP.CN for current national currency."
    },
    "Revenus des facteurs reçus de l'extérieur nets": {
        "type": "derived",
        "function": lambda df: df['NY.GNP.MKTP.CD'] - df['NY.GDP.MKTP.CD'],
        "required_codes": ["NY.GNP.MKTP.CD", "NY.GDP.MKTP.CD"],
        "description": "Net factor income from abroad. Calculate as GNI minus GDP."
    },
    "REVENU NATIONAL": {
        "type": "direct",
        "code": "NY.GNP.MKTP.CD",
        "description": "Gross National Income (current US$). Use NY.GNP.MKTP.CN for current national currency."
    },
    "Autres transferts courants extérieurs nets": {
        "type": "direct",
        "code": "NY.TRF.NCTR.CD",
        "description": "Net current transfers from abroad (current US$), including remittances and grants."
    },
    "REVENU NATIONAL DISPONIBLE BRUT": {
        "type": "derived",
        "function": lambda df: df['NY.GNP.MKTP.CD'] + df['NY.TRF.NCTR.CD'],
        "required_codes": ["NY.GNP.MKTP.CD", "NY.TRF.NCTR.CD"],
        "description": "Gross National Disposable Income. Derive by adding net current transfers to GNI."
    },
    "Sociétés non financières": {
        "type": "none",
        "description": "Non-financial corporations (institutional sector). Not available in World Bank data; use national accounts."
    },
    "Institutions financières": {
        "type": "none",
        "description": "Financial institutions (institutional sector). Not available in World Bank data; use national accounts."
    },
    "Administration Publique": {
        "type": "none",
        "description": "Public Administration/General Government. Limited World Bank data (e.g., GC.DOD.TOTL.GD.ZS for debt); use national accounts."
    },
    "Ménages": {
        "type": "none",
        "description": "Households (institutional sector). Not available in World Bank data; use national accounts."
    },
    "Amortissements": {
        "type": "direct",
        "code": "NY.GDP.CFCF.KD",
        "description": "Consumption of fixed capital (constant 2015 US$). Alternatively, derive as NY.GNP.MKTP.CD - NY.NNP.MKTP.CD."
    },
    "REVENU NATIONAL NET": {
        "type": "direct",
        "code": "NY.NNP.MKTP.CD",
        "description": "Net National Income (current US$). Use NY.NNP.MKTP.CN for current national currency."
    },
    "REVENU NATIONAL DISPONIBLE": {
        "type": "derived",
        "function": lambda df: df['NY.NNP.MKTP.CD'] + df['NY.TRF.NCTR.CD'],
        "required_codes": ["NY.NNP.MKTP.CD", "NY.TRF.NCTR.CD"],
        "description": "Net National Disposable Income. Derive by adding net current transfers to NNI."
    },
    "Consommation Finale": {
        "type": "direct",
        "code": "NE.CON.TOTL.CD",
        "description": "Final consumption expenditure (current US$). Use NE.CON.PRVT.CD for households, NE.CON.GOVT.CD for government."
    },
    "EPARGNE NATIONALE (BRUTE)": {
        "type": "direct",
        "code": "NY.GNS.ICTR.CD",
        "description": "Gross national saving (current US$). Represents GNDI minus final consumption."
    },
    "EPARGNE NATIONALE (NETTE)": {
        "type": "derived",
        "function": lambda df: df['NY.GNS.ICTR.CD'] - df['NY.GDP.CFCF.KD'],
        "required_codes": ["NY.GNS.ICTR.CD", "NY.GDP.CFCF.KD"],
        "description": "Net national saving. Derive by subtracting consumption of fixed capital from gross saving."
    },
    "Taux d'épargne par agent": {
        "type": "none",
        "description": "Saving rate by institutional sector. Not available in World Bank data; use national accounts."
    },
    "TAUX D'EPARGNE": {
        "type": "direct",
        "code": "NY.GNS.ICTR.ZS",
        "description": "Gross savings as percentage of GNI."
    },
    "Population (en milliers)": {
        "type": "direct",
        "code": "SP.POP.TOTL",
        "description": "Total population (divide by 1,000 for thousands)."
    },
    "Nombre de ménage (en milliers)": {
        "type": "none",
        "description": "Number of households. Not available in World Bank data; use national surveys."
    },
    "Taille moyenne d'un ménage": {
        "type": "none",
        "description": "Average household size. Not available in World Bank data; derive from national census data."
    },
    "INDICE DES PRIX A LA CONSOMMATION FAMILIALE (IPC; 2015=100)": {
        "type": "direct",
        "code": "FP.CPI.TOTL",
        "description": "Consumer Price Index (2015=100)."
    },
    "REVENU DISPONIBLE BRUT DES MENAGES en MD": {
        "type": "none",
        "description": "Gross Disposable Income of Households (in millions of national currency). Not available in World Bank data; use national accounts."
    },
    "Accroissement": {
        "type": "derived",
        "function": lambda df: df['NY.GDP.MKTP.KD'].pct_change() * 100,
        "required_codes": ["NY.GDP.MKTP.KD"],
        "description": "GDP growth rate (annual %). Use NY.GDP.MKTP.KD.ZS if available or calculate from real GDP."
    },
    "REVENU DISPONIBLE BRUT PAR MENAGE en Dinars courants": {
        "type": "none",
        "description": "Gross Disposable Income per Household (current Dinars). Derive from national accounts divided by number of households."
    },
    "REVENU DISPONIBLE BRUT PAR MENAGE en Dinars de 2015": {
        "type": "none",
        "description": "Gross Disposable Income per Household (2015 Dinars, purchasing power). Deflate nominal household income using FP.CPI.TOTL."
    },
    "GFCF by Institutional Sectors (MD)": {
        "type": "direct",
        "code": "NE.GDI.FTOT.CD",
        "description": "Gross Fixed Capital Formation (current US$, total). Sectoral breakdowns not available in World Bank data."
    },
    "Non-Financial Corporations (NFCs) (GFCF)": {
        "type": "none",
        "description": "GFCF for non-financial corporations. Not available in World Bank data; use national accounts."
    },
    "Financial Institution (GFCF)": {
        "type": "none",
        "description": "GFCF for financial institutions. Not available in World Bank data; use national accounts."
    },
    "Public Administration (GFCF)": {
        "type": "none",
        "description": "GFCF for public administration. Not available in World Bank data; use national accounts."
    },
    "Householders (housing) (GFCF)": {
        "type": "none",
        "description": "GFCF for households (housing). Not available in World Bank data; use national accounts."
    },
    "TOTAL GENERAL (GFCF)": {
        "type": "direct",
        "code": "NE.GDI.FTOT.CD",
        "description": "Total Gross Fixed Capital Formation (current US$). Use NE.GDI.FTOT.CN for national currency."
    },
    "Inflation": {
        "type": "direct",
        "code": "FP.CPI.TOTL.ZG",
        "description": "Inflation, consumer prices (annual %)."
    },
    "RNDB (MD 2015)": {
        "type": "derived",
        "function": lambda df: (df['NY.GNP.MKTP.CD'] + df['NY.TRF.NCTR.CD']) / (df['FP.CPI.TOTL'] / 100),
        "required_codes": ["NY.GNP.MKTP.CD", "NY.TRF.NCTR.CD", "FP.CPI.TOTL"],
        "description": "Gross National Disposable Income (2015 prices). Deflate nominal GNDI using CPI."
    },
    "RNDB/Personne": {
        "type": "derived",
        "function": lambda df: (df['NY.GNP.MKTP.CD'] + df['NY.TRF.NCTR.CD']) / df['SP.POP.TOTL'],
        "required_codes": ["NY.GNP.MKTP.CD", "NY.TRF.NCTR.CD", "SP.POP.TOTL"],
        "description": "Gross National Disposable Income per Person (current US$)."
    }
}

# Collect all required indicator codes
required_codes = set()
for indicator in economic_indicators.values():
    if indicator['type'] == 'direct':
        required_codes.add(indicator['code'])
    elif indicator['type'] == 'derived':
        required_codes.update(indicator['required_codes'])

# Filter the DataFrame to include only required indicator codes and years 2018-2025
df_filtered = df[df['Indicator Code'].isin(required_codes) & df['Year'].between(2018, 2025)]

# Check for duplicates in 'Year' and 'Indicator Code'
duplicates = df_filtered[df_filtered.duplicated(subset=['Year', 'Indicator Code'], keep=False)]
if not duplicates.empty:
    print("Warning: Duplicate entries found in the data:")
    print(duplicates[['Year', 'Indicator Code', 'Value']])
    # Aggregate duplicates by taking the mean of 'Value' for each 'Year' and 'Indicator Code'
    df_filtered = df_filtered.groupby(['Year', 'Indicator Code'])['Value'].mean().reset_index()
    print("Duplicates aggregated by taking the mean of 'Value'.")

# Pivot the filtered DataFrame to have years as rows and indicator codes as columns
pivoted_df = df_filtered.pivot(index='Year', columns='Indicator Code', values='Value')

# Create the result DataFrame with years as the index
result_df = pd.DataFrame(index=range(2018, 2026))

# Populate the result DataFrame with indicator values mapped to French names
for french_term, details in economic_indicators.items():
    if details['type'] == 'direct':
        code = details['code']
        # Check if the code exists in the pivoted DataFrame
        if code in pivoted_df.columns:
            result_df[french_term] = pivoted_df[code].reindex(result_df.index)
        else:
            # If the code is missing, fill with NaN
            result_df[french_term] = np.nan
    elif details['type'] == 'derived':
        try:
            # Attempt to compute the derived indicator using the provided function
            result_df[french_term] = details['function'](pivoted_df).reindex(result_df.index)
        except (KeyError, TypeError):
            # If any required code is missing or computation fails, fill with NaN
            result_df[french_term] = np.nan
    else:  # type == 'none'
        # For indicators not available in World Bank data, fill with NaN
        result_df[french_term] = np.nan

# Sort the index to ensure years are in chronological order
result_df = result_df.sort_index()

# Display the first few rows of the result
result_df.head(8)

# Optional: Save the result to a CSV file
result_df.to_csv('tunisia_indicators_mapped_2018_2025.csv')

       Year  Indicator Code         Value
7270   2024  FP.CPI.TOTL.ZG  7.206617e+00
7271   2023  FP.CPI.TOTL.ZG  9.328996e+00
7272   2022  FP.CPI.TOTL.ZG  8.306461e+00
7273   2021  FP.CPI.TOTL.ZG  5.706350e+00
7274   2020  FP.CPI.TOTL.ZG  5.634151e+00
7275   2019  FP.CPI.TOTL.ZG  6.720075e+00
7276   2018  FP.CPI.TOTL.ZG  7.307592e+00
15183  2023  NY.GNP.MKTP.CD  4.721969e+10
15184  2022  NY.GNP.MKTP.CD  4.360672e+10
15185  2021  NY.GNP.MKTP.CD  4.558218e+10
15186  2020  NY.GNP.MKTP.CD  4.109505e+10
15187  2019  NY.GNP.MKTP.CD  4.081334e+10
15188  2018  NY.GNP.MKTP.CD  4.165026e+10
42130  2024  FP.CPI.TOTL.ZG  7.206617e+00
42131  2023  FP.CPI.TOTL.ZG  9.328996e+00
42132  2022  FP.CPI.TOTL.ZG  8.306461e+00
42133  2021  FP.CPI.TOTL.ZG  5.706350e+00
42134  2020  FP.CPI.TOTL.ZG  5.634151e+00
42135  2019  FP.CPI.TOTL.ZG  6.720075e+00
42136  2018  FP.CPI.TOTL.ZG  7.307592e+00
51736  2023     SP.POP.TOTL  1.220043e+07
51737  2022     SP.POP.TOTL  1.211933e+07
51738  2021     SP.POP.TOTL  1.204